# RolloTree Advanced Usage

This notebook covers advanced topics:
1. Comparing impurity criteria (Gini vs Misclassification)
2. Exploring different tree depths
3. Switching between solvers (HiGHS, CBC, Gurobi)
4. Solver configuration options
5. Working with custom datasets

In [ ]:
import pandas as pd
import numpy as np
import time
from rollotree import RollingOCT

In [ ]:
# Load the bundled Wine dataset
train = pd.read_csv("../rollotree/data/train.csv")
test = pd.read_csv("../rollotree/data/test.csv")

X_train = train.drop("y", axis=1)
y_train = train["y"]
X_test = test.drop("y", axis=1)
y_test = test["y"]

print(f"Train: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]} samples, {X_test.shape[1]} features")
print(f"Classes: {sorted(y_train.unique())}")

## 1. Comparing Impurity Criteria

RolloTree supports two impurity criteria for the MIP objective:
- **Gini index** (`criterion="gini"`): Minimizes weighted Gini impurity across leaves
- **Misclassification error** (`criterion="misclassification"`): Minimizes the number of misclassified samples

In [ ]:
results = {}

for criterion in ["gini", "misclassification"]:
    model = RollingOCT(depth=3, criterion=criterion, solver="highs")
    
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    results[criterion] = {
        "train_acc": train_acc,
        "test_acc": test_acc,
        "time": elapsed,
    }
    print(f"{criterion:20s}  train={train_acc:.3f}  test={test_acc:.3f}  time={elapsed:.2f}s")

print()
print("Gini tends to produce better-balanced splits; misclassification directly minimizes errors.")

## 2. Exploring Different Tree Depths

- **Depth 2**: Base case — solves a single OCT-2 MIP
- **Depth 3+**: Rolling subtree expansion — iteratively expands misclassified leaves

Deeper trees are more expressive but take longer to train.

In [ ]:
depth_results = {}

for depth in [2, 3, 4]:
    model = RollingOCT(depth=depth, criterion="gini", solver="highs")
    
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    depth_results[depth] = {
        "train_acc": train_acc,
        "test_acc": test_acc,
        "time": elapsed,
        "model": model,
    }
    print(f"Depth {depth}: train={train_acc:.3f}  test={test_acc:.3f}  time={elapsed:.2f}s")

In [ ]:
# Per-depth breakdown from the depth-4 model
model_d4 = depth_results[4]["model"]

print("Per-depth statistics from the depth-4 model:")
print("-" * 55)
for d, result in sorted(model_d4.depth_results_.items()):
    print(
        f"  Depth {d}: "
        f"train_acc={result.training_accuracy:.3f}, "
        f"test_acc={result.test_accuracy:.3f}, "
        f"time={result.elapsed_time:.2f}s"
    )
print()
print("Training accuracy is non-decreasing as depth increases.")

## 3. Switching Between Solvers

RolloTree supports three solver backends through PuLP:

| Solver | Install | Notes |
|--------|---------|-------|
| `"highs"` | Included (`highspy`) | Best open-source MIP solver. Default. |
| `"cbc"` | Bundled with PuLP | Fallback open-source solver. |
| `"gurobi"` | `pip install gurobipy` + license | Fastest commercial solver. |

In [ ]:
solvers_to_test = ["highs", "cbc"]

# Uncomment to include Gurobi (requires gurobipy + license):
# solvers_to_test.append("gurobi")

solver_results = {}

for solver in solvers_to_test:
    model = RollingOCT(depth=3, criterion="gini", solver=solver)
    
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    solver_results[solver] = {
        "train_acc": train_acc,
        "test_acc": test_acc,
        "time": elapsed,
    }
    print(f"{solver:8s}  train={train_acc:.3f}  test={test_acc:.3f}  time={elapsed:.2f}s")

## 4. Solver Configuration Options

Fine-tune solver behavior with these parameters:
- `time_limit`: Maximum seconds per depth-2 subproblem (default: 1800)
- `mip_gap`: Acceptable optimality gap, e.g. 0.01 for 1% (default: None = optimal)
- `big_m`: Penalty for empty-leaf splits (default: 99)
- `log_to_console`: Show solver output (default: False)

In [ ]:
# Example: fast solve with relaxed optimality gap
model_fast = RollingOCT(
    depth=3,
    criterion="gini",
    solver="highs",
    time_limit=60,
    mip_gap=0.05,  # accept 5% gap
)

start = time.time()
model_fast.fit(X_train, y_train)
elapsed = time.time() - start

print(f"Fast model: train={model_fast.score(X_train, y_train):.3f}  "
      f"test={model_fast.score(X_test, y_test):.3f}  time={elapsed:.2f}s")

In [ ]:
# Example: verbose solver output
model_verbose = RollingOCT(
    depth=2,
    solver="highs",
    log_to_console=True,  # shows solver progress
)
model_verbose.fit(X_train, y_train)
print(f"\nAccuracy: {model_verbose.score(X_test, y_test):.3f}")

## 5. Working with Custom Datasets

RolloTree requires **binary** (0/1) features. If your data has continuous or categorical features, binarize them first.

The bundled `make_data_binary` helper can do this automatically.

In [ ]:
from rollotree.preprocessing.helpers import make_data_binary

# Synthetic dataset with mixed feature types
np.random.seed(42)
n = 100
raw_data = pd.DataFrame({
    "y": np.random.choice([1, 2, 3], size=n),
    "temperature": np.random.uniform(15.0, 35.0, size=n),
    "humidity": np.random.uniform(0.2, 0.9, size=n),
    "season": np.random.choice(["spring", "summer", "fall", "winter"], size=n),
})

print("Raw data:")
print(raw_data.head())
print(f"Shape: {raw_data.shape}")

In [ ]:
# Binarize: continuous features get median-split, categoricals get one-hot encoded
binary_data = make_data_binary(raw_data)

print(f"Binary data shape: {binary_data.shape}")
print(f"Columns: {list(binary_data.columns[:10])}...")
print()
binary_data.head()

In [ ]:
# Train on the binarized data
X = binary_data.drop("y", axis=1)
y = binary_data["y"]

# Simple train/test split
split = int(0.8 * len(X))
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y.iloc[:split], y.iloc[split:]

model = RollingOCT(depth=3, solver="highs")
model.fit(X_tr, y_tr)

print(f"Train accuracy: {model.score(X_tr, y_tr):.3f}")
print(f"Test accuracy:  {model.score(X_te, y_te):.3f}")

## 6. Inspecting the Tree Structure

After fitting, `model.tree_` gives access to the full tree for inspection.

In [ ]:
# Use the wine depth-4 model
tree = model_d4.tree_

print(f"Tree depth: {tree.depth}")
print(f"Number of branch nodes: {len(tree.branch_nodes)}")
print(f"Number of leaf nodes:   {len(tree.leaf_nodes)}")
print()

# Show branch node splits
print("Branch nodes:")
for nid, node in sorted(tree.branch_nodes.items()):
    feat = node.feature_index if node.feature_index is not None else "none"
    print(f"  Node {nid:2d}: splits on feature {feat}")

print()

# Show leaf predictions
print("Leaf nodes:")
for lid, leaf in sorted(tree.leaf_nodes.items()):
    cls = leaf.predicted_class if leaf.predicted_class is not None else "none"
    status = " (pruned)" if leaf.is_pruned else ""
    print(f"  Leaf {lid:2d}: predicts class {cls}{status}")

## Summary

| Feature | Options |
|---------|--------|
| Criterion | `"gini"`, `"misclassification"` |
| Solver | `"highs"` (default), `"cbc"`, `"gurobi"` |
| Depth | Any integer >= 2 |
| Time limit | Seconds per subproblem (default: 1800) |
| MIP gap | Optimality tolerance (default: None = exact) |

For basic usage, see **01_quickstart.ipynb**.